# 1. Data Processing

1. Importing All The Nesessary Libraries.
2. Uploading the Dataset and Merging into 1 Dataset.
3. Checking Merged Dataset
4. Cleaning Steps

## 1. Importing All The Nesessary Libraries

In [67]:
# importing all the Necessary Libraries

import numpy as np
import pandas as pd
from pathlib import Path
import os

##  2. Uploading the Dataset and Merging into 1 Dataset

In [68]:
# Create an empty list to merge all the datasets
all_patients_list = []
#Load your list of ID's from the csv file
patient_ids = pd.read_csv("T1DM_patient_sleep_demographics_with_race.csv")['Patient_ID'].tolist()

# Slice the list to take only the first 5 Patient IDs
first5patients = patient_ids[:5]

# Loading file path to access the files
patientfiles = Path("HUPA-UC Diabetes Dataset1-5")

patient_files = sorted(patientfiles.glob("*.csv"))

#Loop for processing all the files and merging into 1 file 
for f in patient_files:
    df = pd.read_csv(f, sep = ";", index_col = None, header= 0)

    # Creating a new column to add Patient ID's
    df['patient_id'] = f.stem
    all_patients_list.append(df)

#------------MERGING EVERYTHING-------------
merged_data = pd.concat (all_patients_list, ignore_index = True)
        
      

In [69]:
df.head()

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id
0,2018-07-09T14:20:00,184.838710,4.94566,83.906977,0.0,0.075,0.0,0.0,HUPA0005P
1,2018-07-09T14:25:00,188.064516,5.28674,81.858156,0.0,0.075,0.0,0.0,HUPA0005P
2,2018-07-09T14:30:00,191.290323,4.68985,79.834646,0.0,0.075,0.0,0.0,HUPA0005P
3,2018-07-09T14:35:00,194.516129,8.44173,85.146552,77.0,0.075,0.0,0.0,HUPA0005P
4,2018-07-09T14:40:00,197.741935,5.28674,83.511811,0.0,0.075,0.0,0.0,HUPA0005P


## 3. Checking Merged Dataset 

In [70]:
df.shape

(3858, 9)

In [71]:
df.describe()

,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
count,3858.000000,3858.000000,3858.000000,3858.000000,3858.000000,3858.000000,3858.000000
mean,147.698624,5.554286,84.490902,13.479264,0.066450,0.026102,0.022939
std,49.791133,2.926070,10.991375,46.315006,0.014355,0.221013,0.209460
min,40.000000,4.263500,32.407773,0.000000,0.000000,0.000000,0.000000
25%,109.333333,4.263500,79.875242,0.000000,0.062500,0.000000,0.000000
50%,144.000000,4.263500,85.563429,0.000000,0.070833,0.000000,0.000000
75%,180.666667,5.286740,91.532109,0.000000,0.075000,0.000000,0.000000
max,379.000000,28.139100,124.175819,597.000000,0.083333,4.200000,4.000000


## 4. Cleaning Steps

1. Round numeric columns
2. Check the Datatypes
3. Convert Datatypes
4. Standardizing Time Zone
5. Sorting Our Data For Better Understanding
6. Removing the Duplicates
7. Check the Outliers
    1. Glucose
    2. Calories
    3. Heart Rate

## 1. Rounding All The Numeric Column Values 

In [72]:
    # 1. Round numeric columns
    df = df.round(3)
    

## 2. Check DataTypes

In [73]:
 # 2. Convert time (Fixed the 'df1' typo here)
df.dtypes
   

time                       object
glucose                   float64
calories                  float64
heart_rate                float64
steps                     float64
basal_rate                float64
bolus_volume_delivered    float64
carb_input                float64
patient_id                 object
dtype: object

## 3. Converting Time column Datatype

In [74]:
df['time'] = pd.to_datetime(df['time'])

## 4. Standardizing Time Zone 

In [75]:
## Standardizing time zones
df['time']=pd.to_datetime(df['time'])
# Convert to datetime and strip timezone info
df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)


## 5. Sorting our data for better understanding

In [76]:
# sorting our data according to Patient ID and time
df = df.sort_values(['patient_id', 'time'])

## 6. Removing the Duplicates

In [77]:
# remove duplicates 

df = df.drop_duplicates(subset=['patient_id', 'time'], keep='first').reset_index(drop=True)

## Outliers

We are using **Interquartile Range(IQR) Method**, which is a standard statistical technique for finding data points that fall significantly outside the central range of a dataset.
    This is a way to find outliers- data point that are unsually high or low compared to the rest of the data- by focusing on the middle 50% of our data

In [ ]:
# These are the columns to check for outliers
numeric_cols = [
    'glucose',
    'calories',
    'heart_rate',
    'steps',
    'basal_rate',
    'bolus_volume_delivered',
    'carb_input'
]

# to finds the 25th percentil (Q1) and 75th percentile(Q3) for each health metric 
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# It creates "Fences" to define the range of normal values.
outliers = (df[numeric_cols] < (Q1 - 1.5 * IQR)) | \
           (df[numeric_cols] > (Q3 + 1.5 * IQR))

# Counts how many true values exist in each column, giving you a total outlier count for each metric.
outliers.sum()

## Display Outlier Preview

## Saving The Cleaned Dataset

In [80]:
##----------------SAVE TE CLEANED FILE----------------------------
cleaned_data.to_csv("cleaned_file.csv", index = False)